# Project 1 - Celebrity Detection
## Milestone 2: Synthetic Multi-Face Dataset Construction

**IE 7615 - Deep Learning in Vision Systems**
**David Fung**

Milestone 2 is the second graded checkpoint for Project 1, a discriminative computer-vision pipeline for celebrity identification and detection. This milestone builds the **synthetic multi-celebrity detection dataset** that Milestone 3 trains YOLOv8 on. It composites 2-5 celebrity faces (the 5 identities selected in Milestone 1) onto varied synthetic backgrounds, annotates every face with a tight YOLO-format bounding box, documents a detection-appropriate augmentation strategy, and produces a clean 70/15/15 train/val/test split.

This is the bridge from single-face classification (Milestone 1) to object detection (Milestone 3). Strong, well-annotated, well-split data at this checkpoint directly drives every downstream metric (mAP, IoU, precision, recall) reported later. It runs in parallel with Module 4's RNN content; Project 1 itself contains no RNN component.

## Design Decisions

| Parameter | Choice | Rationale |
|---|---|---|
| Canvas size | 640x640 | Matches the YOLOv8 `imgsz=640` used in Milestone 3 training. |
| Faces per image | 2-5 (random) | Satisfies "two or more identities per image" and adds compositional difficulty. |
| Identities per image | always distinct | Every frame is genuinely multi-celebrity. |
| Backgrounds | 50% gradient / 30% blurred noise / 20% blurred face | Variety prevents the model memorising a background template or shortcut. |
| Face scale | 15-35% of canvas height | Large enough to be reliably detected, small enough for real composition variety. |
| Face rotation | +/-10 degrees | Small, realistic pose variation; boxes are recomputed after rotation. |
| Brightness jitter | 0.85-1.15 | Mild lighting variation for robustness. |
| Split | 70 / 15 / 15 | Standard; every identity appears in every split. |
| Random seed | 42 | Fully reproducible pipeline. |

In [ ]:
import os
import random
import shutil
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from PIL import Image, ImageEnhance, ImageFilter
from sklearn.model_selection import train_test_split

# Deterministic seeding (Milestone 1 convention: GLOBAL_SEED = 42)
GLOBAL_SEED = 42
random.seed(GLOBAL_SEED)
np.random.seed(GLOBAL_SEED)
try:
    import tensorflow as tf
    tf.random.set_seed(GLOBAL_SEED)
    print("TensorFlow seeded:", tf.__version__)
except Exception:
    pass
print("RNG seeded for reproducibility (seed=%d)" % GLOBAL_SEED)

In [ ]:
# --- Dataset constants (consistent with Milestone 1) ---
SYNTH_IMG_SIZE = (640, 640)
IMG_W, IMG_H = SYNTH_IMG_SIZE
NUM_IMAGES = 200          # total synthetic images to generate
MIN_FACES, MAX_FACES = 2, 5
CLASS_NAMES = ['7', '797', '2619', '4428', '7007']
NUM_CLASSES = len(CLASS_NAMES)
TRAIN_RATIO, VAL_RATIO, TEST_RATIO = 0.70, 0.15, 0.15

def _resolve_project_root():
    # Walk upward from the current working directory until data/selected_images is found.
    # This makes the notebook work whether it is launched from the repo root or from Milestone_2/.
    cur = os.getcwd()
    for _ in range(8):
        if os.path.isdir(os.path.join(cur, 'data', 'selected_images')):
            return cur
        parent = os.path.dirname(cur)
        if parent == cur:
            break
        cur = parent
    raise FileNotFoundError("Could not find data/selected_images from cwd=%s" % os.getcwd())

ROOT = _resolve_project_root()
FACE_POOL_DIR = os.path.join(ROOT, 'data', 'selected_images')
OUTPUT_DIR = os.path.join(ROOT, 'data', 'synthetic_multi_face')
print("Project root :", ROOT)
print("Face pool    :", FACE_POOL_DIR)
print("Output dir   :", OUTPUT_DIR)

### Task 1 - Load the Face Pool

Milestone 1 selected and cropped a small, high-quality set of faces for the 5 target celebrities and stored them in `data/selected_images/<identity>/`. This cell builds the identity-to-face-path mapping used by the generator.

The identity names (`7`, `797`, `2619`, `4428`, `7007`) are the CelebA identity IDs carried over from Milestone 1. They are used verbatim as the YOLO class names so that the detection dataset stays consistent with the classification dataset and with `data.yaml`.

In [ ]:
identity_pools = {}
for class_name in CLASS_NAMES:
    class_dir = os.path.join(FACE_POOL_DIR, class_name)
    if not os.path.isdir(class_dir):
        raise FileNotFoundError("Missing identity folder: %s" % class_dir)
    paths = sorted(
        os.path.join(class_dir, f)
        for f in os.listdir(class_dir)
        if f.lower().endswith(('.jpg', '.jpeg', '.png'))
    )
    if not paths:
        raise FileNotFoundError("No face images in %s" % class_dir)
    identity_pools[class_name] = paths

name_to_idx = {name: i for i, name in enumerate(CLASS_NAMES)}
total_faces = sum(len(v) for v in identity_pools.values())
print("Loaded face pool (%d faces, %d identities):" % (total_faces, len(identity_pools)))
for name, paths in identity_pools.items():
    print("  class %d  %-4s  %3d faces" % (name_to_idx[name], name, len(paths)))

### Task 2 - Synthetic Background Generation

Using a single flat colour for every background would let the detector memorise a background "template" instead of learning the face. Each frame is therefore drawn from one of three background families:

- **Linear gradient (50%)** - two random muted colours blended top-to-bottom.
- **Blurred noise field (30%)** - a Gaussian-blurred random-noise image, mimicking an out-of-focus scene.
- **Blurred face (20%)** - a random face crop, heavily blurred and darkened so it reads as a defocused background rather than a second (distracting) face.

All three keep the foreground faces as the only sharp objects in the frame.

In [ ]:
def _linear_gradient(w, h, c1, c2):
    # vertical gradient: colour varies from c1 (top) to c2 (bottom), constant across width
    t = np.linspace(0.0, 1.0, h)[:, None]
    t = np.broadcast_to(t, (h, w))
    c1 = np.asarray(c1, float).reshape(1, 1, 3)
    c2 = np.asarray(c2, float).reshape(1, 1, 3)
    t3 = t[..., None]
    arr = c1 * (1.0 - t3) + c2 * t3
    return Image.fromarray(arr.astype(np.uint8), 'RGB')

def _bg_gradient(w, h):
    c1 = [random.randint(25, 130) for _ in range(3)]
    c2 = [random.randint(25, 130) for _ in range(3)]
    return _linear_gradient(w, h, c1, c2)

def _bg_noise(w, h):
    arr = np.random.randint(50, 160, (h, w, 3), dtype=np.uint8)
    return Image.fromarray(arr, 'RGB').filter(ImageFilter.GaussianBlur(radius=random.randint(20, 60)))

def _bg_blur_face(w, h):
    all_paths = [p for paths in identity_pools.values() for p in paths]
    if all_paths:
        img = Image.open(random.choice(all_paths)).convert('RGB').resize((w, h))
        img = img.filter(ImageFilter.GaussianBlur(radius=random.randint(30, 80)))
        return ImageEnhance.Brightness(img).enhance(random.uniform(0.5, 0.8))
    return _bg_gradient(w, h)

def generate_background(w, h):
    r = random.random()
    if r < 0.50:
        return _bg_gradient(w, h)
    if r < 0.80:
        return _bg_noise(w, h)
    return _bg_blur_face(w, h)

# Preview: one of each background family
fig, ax = plt.subplots(1, 3, figsize=(15, 5))
for a, fn, title in zip(ax, [_bg_gradient, _bg_noise, _bg_blur_face],
                        ['gradient', 'blurred noise', 'blurred face']):
    a.imshow(fn(IMG_W, IMG_H)); a.axis('off'); a.set_title(title)
plt.suptitle('Task 2 - the three synthetic background families')
plt.tight_layout(); plt.show()

### Task 3 - Face Placement Algorithm

Faces are composited with realistic, randomised parameters:

- **Scale** - each face height is 15-35% of the canvas, so faces span a realistic range of apparent sizes.
- **Rotation** - a small random rotation (+/-10 degrees) is applied before pasting, giving mild pose variation.
- **Brightness** - a mild brightness jitter (0.85-1.15) simulates inconsistent lighting.
- **Position** - the face is placed anywhere inside a 5% safety margin so it stays fully in frame.

Because PIL expands the canvas when an image is rotated, the YOLO box is computed from the *post-rotation* size and position, so every box is a tight, correctly localised rectangle even for rotated faces. Identities per image are drawn **without replacement**, so no image contains the same celebrity twice.

In [ ]:
def place_faces(background, num_faces):
    canvas = background.convert('RGBA').copy()
    W, H = canvas.size
    boxes = []
    names = list(identity_pools.keys())
    chosen = random.sample(names, min(num_faces, len(names)))   # distinct identities
    margin = 0.05
    for identity in chosen:
        face = Image.open(random.choice(identity_pools[identity])).convert('RGB')
        target_h = int(H * random.uniform(0.15, 0.35))
        target_w = max(1, int(target_h * face.width / face.height))
        face = face.resize((target_w, target_h))
        if random.random() < 0.5:
            face = face.transpose(Image.FLIP_LEFT_RIGHT)
        face = ImageEnhance.Brightness(face).enhance(random.uniform(0.85, 1.15))
        face = face.convert('RGBA')
        face = face.rotate(random.uniform(-10, 10), expand=True, resample=Image.BICUBIC)
        rw, rh = face.size
        if rw > W - 2:
            s = (W - 2) / float(rw)
            face = face.resize((max(1, int(rw * s)), max(1, int(rh * s))))
            rw, rh = face.size
        lo_x, hi_x = int(margin * W), W - rw - int(margin * W)
        lo_y, hi_y = int(margin * H), H - rh - int(margin * H)
        hi_x = max(hi_x, lo_x); hi_y = max(hi_y, lo_y)
        x = random.randint(lo_x, hi_x)
        y = random.randint(lo_y, hi_y)
        canvas.paste(face, (x, y), face)   # face's own alpha channel = composite mask
        boxes.append((name_to_idx[identity], (x + rw / 2.0) / W, (y + rh / 2.0) / H, rw / W, rh / H))
    return canvas.convert('RGB'), boxes

### Task 4 - Generate the Synthetic Dataset

Each composite frame is written to a staging area together with its YOLO label file. The label stores one line per face:

    <class_index> <cx> <cy> <w> <h>

where all coordinates are normalised to [0, 1] (YOLO format). After all `NUM_IMAGES` frames are written, Task 7 splits the set into train/val/test and the following code cell writes `data.yaml`.

In [ ]:
# Start from a clean staging area so re-runs are reproducible
staging = os.path.join(OUTPUT_DIR, 'staging')
for sub in ('images', 'labels'):
    d = os.path.join(staging, sub)
    if os.path.isdir(d):
        shutil.rmtree(d)
os.makedirs(os.path.join(staging, 'images'), exist_ok=True)
os.makedirs(os.path.join(staging, 'labels'), exist_ok=True)

manifest = []
for img_idx in range(NUM_IMAGES):
    n_faces = random.randint(MIN_FACES, MAX_FACES)
    bg = generate_background(IMG_W, IMG_H)
    final_img, boxes = place_faces(bg, n_faces)
    stem = "synthetic_%04d" % img_idx
    img_path = os.path.join(staging, 'images', stem + '.jpg')
    lbl_path = os.path.join(staging, 'labels', stem + '.txt')
    final_img.save(img_path, quality=90)
    with open(lbl_path, 'w') as f:
        for (ci, cx, cy, bw, bh) in boxes:
            f.write("%d %.6f %.6f %.6f %.6f\n" % (ci, cx, cy, bw, bh))
    manifest.append({
        'index': img_idx,
        'image': img_path,
        'label': lbl_path,
        'n_faces': len(boxes),
        'classes': [CLASS_NAMES[b[0]] for b in boxes],
    })
    if (img_idx + 1) % 50 == 0:
        print("  generated %d / %d" % (img_idx + 1, NUM_IMAGES))

print("Generated %d synthetic images -> %s" % (len(manifest), staging))

### Task 5 - Visual Verification

A sample of generated frames is rendered with their YOLO bounding boxes overlaid. Every face should be tightly framed, and the boxes should be localised correctly even for rotated faces.

In [ ]:
def annotate_ax(ax, img_path, label_path):
    img = Image.open(img_path)
    w, h = img.size
    ax.imshow(img)
    with open(label_path) as f:
        for line in f.read().strip().splitlines():
            parts = line.split()
            if not parts:
                continue
            ci = int(parts[0])
            cx, cy, bw, bh = (float(v) for v in parts[1:5])
            x1, y1 = (cx - bw / 2.0) * w, (cy - bh / 2.0) * h
            x2, y2 = (cx + bw / 2.0) * w, (cy + bh / 2.0) * h
            ax.add_patch(mpatches.Rectangle((x1, y1), x2 - x1, y2 - y1,
                                            fill=False, edgecolor='red', lw=2))
            ax.text(x1, max(2, y1 - 8), CLASS_NAMES[ci], color='red',
                    fontsize=9, fontweight='bold')
    ax.axis('off')

n_show = min(8, len(manifest))
sample = random.sample(manifest, n_show)
fig, axes = plt.subplots(2, 4, figsize=(20, 10))
for ax, m in zip(np.atleast_1d(axes).flatten(), sample):
    annotate_ax(ax, m['image'], m['label'])
    ax.set_title("%d faces: %s" % (m['n_faces'], ', '.join(m['classes'])), fontsize=9)
plt.suptitle("Synthetic multi-face images with YOLO bounding boxes", fontsize=13)
plt.tight_layout()
plt.show()

### Task 6 - Detection Augmentation Strategy

The synthetic images already carry real variation (background family, scale, rotation, brightness, position). The additional augmentation below is applied by YOLOv8 during Milestone 3 training and is chosen to be **detection-appropriate** - it preserves the meaning of bounding boxes rather than destroying them:

| Augmentation | Setting | Why it is safe for detection |
|---|---|---|
| Horizontal flip | `hflip=0.5` | Mirrors the frame; boxes mirror symmetrically. |
| Mosaic | `mosaic=1.0` | Pastes 4 images into one canvas and rescales boxes - exactly what a crowded multi-face scene needs. |
| HSV jitter | h=0.015, s=0.7, v=0.4 | Small colour/lighting shifts; boxes are unaffected. |
| Random rotation | `degrees=+/-5` | Mild pose variation; boxes rotate with the frame. |
| Random scaling | `scale=0.5` | Zooms in/out; boxes rescale. |
| Mixup | `mixup=0.1` | Low-rate blend to avoid blurring box edges. |
| Copy-paste | `copypaste=0.1` | Injects extra faces with their boxes - raises recall on small/occluded faces. |

Large rotations or aggressive geometric warps are deliberately **avoided**, because they stretch and skew the tight face boxes and make them inconsistent with the YOLO format.

### Task 7 - Train / Val / Test Split

The full set is split 70% train / 15% validation / 15% test with a fixed seed, so the split is reproducible and no image appears in more than one partition. Because every identity appears in every split, the split is identity-balanced.

In [ ]:
# --- reset the split dirs so only the current generation remains ---
for split_name in ('train', 'val', 'test'):
    for kind in ('images', 'labels'):
        d = os.path.join(OUTPUT_DIR, kind, split_name)
        if os.path.isdir(d):
            shutil.rmtree(d)

# --- 70 / 15 / 15 split (reproducible) ---
train_list, val_test_list = train_test_split(
    manifest, test_size=(VAL_RATIO + TEST_RATIO), random_state=GLOBAL_SEED
)
val_list, test_list = train_test_split(
    val_test_list, test_size=TEST_RATIO / (VAL_RATIO + TEST_RATIO), random_state=GLOBAL_SEED
)

def _split_identity_dist(items):
    counts = {n: 0 for n in CLASS_NAMES}
    for m in items:
        for c in m['classes']:
            counts[c] += 1
    return counts

for split_name, items in [('train', train_list), ('val', val_list), ('test', test_list)]:
    img_dir = os.path.join(OUTPUT_DIR, 'images', split_name)
    lbl_dir = os.path.join(OUTPUT_DIR, 'labels', split_name)
    os.makedirs(img_dir, exist_ok=True)
    os.makedirs(lbl_dir, exist_ok=True)
    for m in items:
        shutil.copy2(m['image'], os.path.join(img_dir, os.path.basename(m['image'])))
        shutil.copy2(m['label'], os.path.join(lbl_dir, os.path.basename(m['label'])))
    dist = _split_identity_dist(items)
    print("%5s : %3d images | identity-face counts: %s" %
          (split_name, len(items), {n: dist[n] for n in CLASS_NAMES}))

# drop the staging area so only the split dataset remains
if os.path.isdir(staging):
    shutil.rmtree(staging)

# --- data.yaml ---
yaml_lines = [
    "# Project 1 - Synthetic Multi-Celebrity Detection Dataset",
    "# Auto-generated by Milestone 2 (celebrity_detection)",
    "path: %s" % OUTPUT_DIR,
    "train: images/train",
    "val: images/val",
    "test: images/test",
    "",
    "nc: %d" % NUM_CLASSES,
    "names:",
]
for i, name in enumerate(CLASS_NAMES):
    yaml_lines.append("  %d: %s" % (i, name))
yaml_content = "\n".join(yaml_lines) + "\n"
with open(os.path.join(OUTPUT_DIR, 'data.yaml'), 'w') as f:
    f.write(yaml_content)
print("\nWrote data.yaml:")
print(yaml_content)

## Milestone 2 - Deliverables Summary

- **Synthetic multi-celebrity dataset**: 200 composite frames at 640x640, each with 2-5 distinct celebrity faces from the 5 Milestone 1 identities.
- **YOLO annotations**: every face has a tight, normalised bounding box (`<class> <cx> <cy> <w> <h>`), recomputed correctly for rotated faces.
- **Augmentation strategy**: a documented, detection-appropriate augmentation plan for Milestone 3 YOLOv8 training.
- **Clean split**: reproducible 70 / 15 / 15 train/val/test, identity-balanced, no image leakage.
- **`data.yaml`**: ready for `YOLO("yolov8n.pt").train(data=".../data.yaml", ...)` in Milestone 3.

The dataset lives in the repository under `data/synthetic_multi_face/`. Re-running this notebook (seed `42`) reproduces the full pipeline end-to-end.